In [ ]:
import torch
import torchvision
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import os

# 1. Path to your extracted dataset (Make sure it points to the folder containing 'train' and 'validation')
DATA_ROOT = "C:/Users/MSI/Desktop/Rice Disease AI/RiceLeafsDisease"
TRAIN_DIR = os.path.join(DATA_ROOT, "train")
VAL_DIR = os.path.join(DATA_ROOT, "validation")

# 2. Data Augmentation for training, pure processing for validation
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),   
    transforms.RandomRotation(degrees=15),    
    transforms.ColorJitter(brightness=0.2),   
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),                    
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 3. Load directly from the pre-split folders
train_dataset = datasets.ImageFolder(root=TRAIN_DIR, transform=train_transform)
val_dataset = datasets.ImageFolder(root=VAL_DIR, transform=val_transform)

# 4. Create DataLoaders
BATCH_SIZE = 32 # We can try 32 since MobileNet is lightweight, or keep 16 if you prefer
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"Training images loaded: {len(train_dataset)}")
print(f"Validation images loaded: {len(val_dataset)}")
print(f"Detected Classes ({len(train_dataset.classes)}): {train_dataset.classes}")

Total images: 120
Detected Classes: ['Bacterial leaf blight', 'Brown spot', 'Leaf smut']
Data pipeline upgraded with Augmentation!


In [9]:
import torch.nn as nn
from torchvision.models import mobilenet_v3_large, MobileNet_V3_Large_Weights

# 1. Use your MSI Cyborg's GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 2. Load pre-trained MobileNetV3 weights
weights = MobileNet_V3_Large_Weights.DEFAULT
model = mobilenet_v3_large(weights=weights)

# 3. Change the last layer to match your number of rice classes
num_classes = len(full_dataset.classes)
in_features = model.classifier[3].in_features
model.classifier[3] = nn.Linear(in_features, num_classes)

# 4. Push the model to your GPU
model = model.to(device)

print(f"Model successfully loaded and configured for {num_classes} classes!")

Using device: cuda
Model successfully loaded and configured for 3 classes!


In [10]:
import torch.optim as optim

# 1. Loss Function (How wrong is the AI's guess?)
criterion = nn.CrossEntropyLoss()

# 2. Optimizer (How the AI adjusts its brain to get smarter)
# We use a small learning rate (lr) so it doesn't break the pre-trained weights
optimizer = optim.Adam(model.parameters(), lr=0.001)

print("Optimizer and Loss Function are ready!")

Optimizer and Loss Function are ready!


In [11]:
import time

EPOCHS = 10

print("Starting Training on GPU...")
start_time = time.time()

for epoch in range(EPOCHS):
    model.train() # Set model to training mode
    running_loss = 0.0
    correct_train = 0
    total_train = 0
    
    # --- TRAINING PHASE ---
    for images, labels in train_loader:
        # Move data to your MSI GPU
        images, labels = images.to(device), labels.to(device)
        
        # Clear old gradients
        optimizer.zero_grad()
        
        # Forward pass: Get predictions
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        # Backward pass: Calculate updates
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * images.size(0)
        
        # Calculate training accuracy
        _, predicted = torch.max(outputs, 1)
        total_train += labels.size(0)
        correct_train += (predicted == labels).sum().item()
        
    epoch_loss = running_loss / len(train_dataset)
    train_acc = (correct_train / total_train) * 100
    
    # --- VALIDATION PHASE ---
    model.eval() # Set model to evaluation mode
    correct_val = 0
    total_val = 0
    
    with torch.no_grad(): # Turn off gradient calculation for speed/memory
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            total_val += labels.size(0)
            correct_val += (predicted == labels).sum().item()
            
    val_acc = (correct_val / total_val) * 100
    
    print(f"Epoch [{epoch+1}/{EPOCHS}] -> "
          f"Train Loss: {epoch_loss:.4f} | "
          f"Train Acc: {train_acc:.2f}% | "
          f"Val Acc: {val_acc:.2f}%")

total_time = time.time() - start_time
print(f"\n🎉 Training complete in {total_time:.2f} seconds!")

Starting Training on GPU...
Epoch [1/10] -> Train Loss: 0.9548 | Train Acc: 53.12% | Val Acc: 75.00%
Epoch [2/10] -> Train Loss: 0.3476 | Train Acc: 90.62% | Val Acc: 79.17%
Epoch [3/10] -> Train Loss: 0.1250 | Train Acc: 94.79% | Val Acc: 87.50%
Epoch [4/10] -> Train Loss: 0.1810 | Train Acc: 93.75% | Val Acc: 91.67%
Epoch [5/10] -> Train Loss: 0.3952 | Train Acc: 89.58% | Val Acc: 95.83%
Epoch [6/10] -> Train Loss: 0.1682 | Train Acc: 93.75% | Val Acc: 87.50%
Epoch [7/10] -> Train Loss: 0.0990 | Train Acc: 96.88% | Val Acc: 91.67%
Epoch [8/10] -> Train Loss: 0.0584 | Train Acc: 98.96% | Val Acc: 87.50%
Epoch [9/10] -> Train Loss: 0.0625 | Train Acc: 98.96% | Val Acc: 87.50%
Epoch [10/10] -> Train Loss: 0.0082 | Train Acc: 100.00% | Val Acc: 79.17%

🎉 Training complete in 75.58 seconds!


In [12]:
torch.save(model.state_dict(), "rice_disease_mobilenetv3.pth")
print("Model saved successfully!")

Model saved successfully!
